# 03 – Price distribution across cities

- **Research Question:** How are Airbnb listing prices distributed across major tourist destinations? Are prices normally distributed or heavily right-skewed by luxury listings? Which cities show higher price dispersion, and how do medians compare to arithmetic means when evaluating true market levels?
- **Variables:** `price`
- **Statistical Tools:**
  - **Measures of Central Tendency:** Sample mean vs Sample median.
  - **Measures of Dispersion:** Sample standard deviation, Interquartile Range, and Relative Dispersion.
  - **Measures of Shape:** Fisher-Pearson Skewness and Kurtosis.
  - **Outlier Detection:** Tukey's upper fence rule: $\text{Outlier} > Q_3 + 1.5 \cdot IQR$.
  - **Visualizations:**
    - Distribution histograms with kernel density estimates and explicit mean vs median comparison lines.
    - Boxplots showing medians, quartiles ($Q_1, Q_3$), whiskers, and outlier listings, with a focused head-to-head comparison between Barcelona and Madrid (both in EUR).

In [ ]:
from importlib.metadata import distributions
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.cities import CITIES
from src.load import load_all

cols = ['id', 'price', 'room_type', 'latitude', 'longitude', 'neighbourhood_cleansed']
data = load_all(usecols=cols)

data = data.dropna(subset=['price']).copy()

# Adding display names and currency identifiers
data['city_name'] = data['city'].map({k: v.name for k, v in CITIES.items()})
data['currency'] = data['city'].map({k: v.currency for k, v in CITIES.items()})

# Benchmark exchange rates to EUR
fx_to_eur = {
    'EUR': 1.00,
    'USD': 0.86,
    'JPY': 0.0055,
    'MXN': 0.05
}
data['fx_to_eur'] = data['currency'].map(fx_to_eur)
data['price_eur'] = data['price'] * data['fx_to_eur']

num_offers = len(data)
num_cities = data['city'].nunique()
print(f"Loaded {num_offers:,} valid price offers across {num_cities} cities.")

In [ ]:
# Statistical measures of central tendency, dispersion, shape, and outliers
stats_records = []

for city_key, city_obj in CITIES.items():
    s = data[data['city'] == city_key]['price']
    
    # Central tendency
    mean_val = s.mean()
    median_val = s.median()
    
    # Dispersion
    std_val = s.std()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    rel_iqr = (iqr / median_val) * 100
    
    # Shape measures
    skew_val = stats.skew(s)
    kurt_val = stats.kurtosis(s)
    
    # Outliers (Tukey's Upper Fence: Q3 + 1.5 * IQR)
    upper_fence = q3 + 1.5 * iqr
    outliers_cnt = (s > upper_fence).sum()
    outliers_pct = (outliers_cnt / len(s)) * 100
    
    stats_records.append({
        'City': city_obj.name,
        'Currency': city_obj.currency,
        'Offers': len(s),
        'Mean': round(mean_val, 2),
        'Median': round(median_val, 2),
        'Mean / Median': round(mean_val / median_val, 2),
        'Std Dev (s)': round(std_val, 2),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Rel IQR (%)': round(rel_iqr, 1),
        'Skewness': round(skew_val, 2),
        'Kurtosis': round(kurt_val, 2),
        'Upper Fence (Q3 + 1.5*IQR)': round(upper_fence, 2),
        'Outliers Count': outliers_cnt,
        'Outliers (%)': round(outliers_pct, 2)
    })

#Comprehensive Price Distribution Statistics
df_stats = pd.DataFrame(stats_records).set_index('City')
df_stats

In [ ]:
# Direct comparison between the two Eurozone cities: Barcelona vs Madrid
bcn_prices = data[data['city'] == 'barcelona']['price']
mad_prices = data[data['city'] == 'madrid']['price']

# Mann-Whitney U test
u_stat, p_val = stats.mannwhitneyu(bcn_prices, mad_prices, alternative='two-sided')

bcn_med = bcn_prices.median()
mad_med = mad_prices.median()
bcn_mean = bcn_prices.mean()
mad_mean = mad_prices.mean()
bcn_iqr = bcn_prices.quantile(0.75) - bcn_prices.quantile(0.25)
mad_iqr = mad_prices.quantile(0.75) - mad_prices.quantile(0.25)
bcn_fence = bcn_prices.quantile(0.75) + 1.5 * bcn_iqr
mad_fence = mad_prices.quantile(0.75) + 1.5 * mad_iqr

spain_comparison = pd.DataFrame([
    {
        'Metric': 'Number of listings',
        'Barcelona': f"{len(bcn_prices):,}",
        'Madrid': f"{len(mad_prices):,}",
        'Difference (BCN vs MAD)': f"{len(mad_prices) - len(bcn_prices):,}"
    },
    {
        'Metric': 'Median Price (€)',
        'Barcelona': f"€{bcn_med:.2f}",
        'Madrid': f"€{mad_med:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_med - mad_med:.2f} (+{(bcn_med / mad_med - 1)*100:.1f}%)"
    },
    {
        'Metric': 'Mean Price (€)',
        'Barcelona': f"€{bcn_mean:.2f}",
        'Madrid': f"€{mad_mean:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_mean - mad_mean:.2f} (+{(bcn_mean / mad_mean - 1)*100:.1f}%)"
    },
    {
        'Metric': 'Interquartile Range - IQR (€)',
        'Barcelona': f"€{bcn_iqr:.2f}",
        'Madrid': f"€{mad_iqr:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_iqr - mad_iqr:.2f}"
    },
    {
        'Metric': 'Upper Outlier Fence (€)',
        'Barcelona': f"€{bcn_fence:.2f}",
        'Madrid': f"€{mad_fence:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_fence - mad_fence:.2f}"
    },
    {
        'Metric': 'Mann-Whitney U Test',
        'Barcelona': f"U = {u_stat:,.0f}",
        'Madrid': f"p = {p_val:.2e}",
        'Difference (BCN vs MAD)': "Statistically significant (p < 0.0001***)"
    }
]).set_index('Metric')

spain_comparison

In [ ]:
# Price Distribution Histograms with Mean vs Median Comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, (city_key, city_obj) in enumerate(CITIES.items()):
    ax = axes[idx]
    s = data[data['city'] == city_key]['price']
    
    # Clip at 99th percentile for visualization to clearly display the core distribution
    p99 = s.quantile(0.99)
    s_vis = s[s <= p99]
    
    mean_val = s.mean()
    median_val = s.median()
    skew_val = stats.skew(s)
    kurt_val = stats.kurtosis(s)
    
    sns.histplot(s_vis, kde=True, ax=ax, color='#1f77b4', bins=40, stat='density', alpha=0.45)
    
    # Mean line (Red dashed)
    ax.axvline(mean_val, color='#d62728', linestyle='--', linewidth=2, label=f'Mean: {mean_val:,.1f}')
    
    # Median line (Dark blue solid)
    ax.axvline(median_val, color='#003f5c', linestyle='-', linewidth=2, label=f'Median: {median_val:,.1f}')
    
    ax.set_title(f"{city_obj.name} ({city_obj.currency})\nSkewness = {skew_val:.2f} | Kurtosis = {kurt_val:.1f}", 
                 fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(f"Price ({city_obj.currency})", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.legend(loc='upper right', frameon=True, fontsize=9)

plt.suptitle("Figure 1: Airbnb Price Distributions Across All 6 Cities (Clipped at 99th Percentile)\nDemonstrating Strong Right-Skewness: Mean > Median Everywhere", 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study1_price_distribution_histograms.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots and outlier identification
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

city_order = ['Barcelona', 'Madrid', 'Los Angeles', 'New York', 'Tokyo', 'Mexico City']
palette_cities = ['#2b5c8f', '#457b9d', '#e63946', '#d62828', '#f4a261', '#2a9d8f']

# All 6 cities in Standardized EUR Benchmark
sns.boxplot(
    data=data, x='city_name', y='price_eur', order=city_order,
    hue='city_name', legend=False,
    palette=palette_cities, ax=ax1, showfliers=True,
    flierprops=dict(marker='o', markersize=2, alpha=0.25, color='#777777')
)
ax1.set_ylim(0, 1000)
ax1.set_title('Price Boxplots (All 6 Cities in Standardized EUR, y <= €1,000)', fontsize=12, fontweight='bold', pad=10)
ax1.set_ylabel('Price (EUR equivalent)', fontsize=11)
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=15)

eur_medians = data.groupby('city_name')['price_eur'].median().reindex(city_order)
for idx, (c_name, med) in enumerate(eur_medians.items()):
    ax1.text(idx, med + 15, f"€{med:.0f}", ha='center', va='bottom', fontsize=9.5, fontweight='bold')

# Head-to-Head Comparison: Barcelona vs Madrid (Both native EUR)
spain_data = data[data['city'].isin(['barcelona', 'madrid'])].copy()

sns.boxplot(
    data=spain_data, x='city_name', y='price', order=['Barcelona', 'Madrid'],
    hue='city_name', legend=False,
    palette=['#2b5c8f', '#457b9d'], ax=ax2, width=0.45,
    flierprops=dict(marker='o', markersize=3, alpha=0.35, color='#d62728')
)
ax2.set_ylim(0, 800)
ax2.set_title('Head-to-Head Comparison: Barcelona vs Madrid (Native EUR)', fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel('Price (EUR)', fontsize=11)
ax2.set_xlabel('')

# Annotate Q1, Median, Q3, and Upper Outlier Fence (Q3 + 1.5 * IQR)
for idx, c_key in enumerate(['barcelona', 'madrid']):
    s = spain_data[spain_data['city'] == c_key]['price']
    q1 = s.quantile(0.25)
    med = s.median()
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    fence = q3 + 1.5 * iqr
    
    # Draw fence line
    ax2.hlines(fence, idx - 0.25, idx + 0.25, color='#d62728', linestyle=':', linewidth=2)
    ax2.text(idx + 0.28, fence, f"Upper Fence: €{fence:.1f}\n($Q_3 + 1.5 \\cdot IQR$)", 
             ha='left', va='center', fontsize=8.5, color='#d62728', fontweight='bold')
    ax2.text(idx, med + 15, f"Median: €{med:.1f}\nIQR: €{iqr:.1f}", 
             ha='center', va='bottom', fontsize=9, fontweight='bold', 
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85))

plt.suptitle("Figure 2: Price Dispersion and Outlier Analysis (Tukey's Fences)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study1_price_boxplots_and_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

## Findings

### 1. Systematic Right-Skewness: Why the Arithmetic Mean is Misleading
- Across all 6 analyzed cities, price distributions are heavily right-skewed, meaning a small number of extremely expensive listings pulls the averages up.
- In every city, the **Sample Mean is substantially higher than the Sample Median** by **+35% in Barcelona**, **+44% in Madrid**, **+71% in Los Angeles**, **+59% in New York**, **+59% in Tokyo**, and **+66% in Mexico City**.
- **Key takeaway:** Luxury villas, penthouses, and overpriced listings create a heavy right tail. Because of this, the arithmetic mean does not represent a "typical" stay. The median and interquartile range are much more reliable, robust measures for analyzing accommodation costs.

### 2. Price Dispersion: Barcelona vs Madrid (Eurozone Direct Comparison)
- Comparing Spain's two largest cities in native EUR provides a clear picture without exchange-rate distortions:
  - **Median level:** Barcelona's median price (€177.67) is +42.1% higher than Madrid's (€125.00).
  - **Price spread:** Barcelona has nearly double the price variation of Madrid (IQR = €196.21 in Barcelona vs €100.26 in Madrid).
  - **Statistical confirmation:** A Mann-Whitney U test confirms that prices in Barcelona are systematically higher than in Madrid ($U = 155,985,046, p < 0.0001$).
  - **Market explanation:** Barcelona's coastal location, high tourism demand, and strict rental regulations concentrate supply into higher-end segments, whereas Madrid features a larger, more evenly distributed corporate rental market.

### 3. Outlier Identification and Global Spread
- Using Tukey's rule ($Q_3 + 1.5 \cdot IQR$), we can clearly separate standard accommodations from ultra-luxury properties:
  - **Barcelona:** Listings priced above €579.82 are outliers (4.39% of the market).
  - **Madrid:** Listings priced above €334.38 are outliers (7.14% of the market).
  - **Los Angeles & New York:** Show the highest relative dispersion (IQR / Me around 118–130%), driven by high-end mansions and large suburban properties, while Madrid has the most compact price distribution (80.2\%).

### 4. Summary & Interview Defense Key Points
- **Why use the median instead of the mean?** The mean is heavily distorted by extreme values. The median shows the true midpoint where half the listings are cheaper and half are more expensive, completely ignoring a €10,000/night penthouse.
- **Which city has greater price dispersion?** Barcelona has significantly greater spread than Madrid, both in absolute terms (€196.21 vs €100.26 IQR) and relative variance. Globally, Los Angeles has the highest relative dispersion (130\%).
- **What role do outliers play?** Outliers aren't data errors - they represent genuine luxury estates. However, keeping them unadjusted in predictive models can skew results, which justifies using logarithmic transformations in later regression steps.

In [ ]:
import plotly.express as px

price_col = 'price_eur' if 'price_eur' in data.columns else 'price'
color_cap = data[price_col].quantile(0.95)

hover_cols = ['room_type']
if 'neighbourhood' in data.columns:
    hover_cols.append('neighbourhood')
elif 'neighbourhood_cleansed' in data.columns:
    hover_cols.append('neighbourhood_cleansed')

fig = px.scatter_geo(
    data,
    lat='latitude',
    lon='longitude',
    color=price_col,
    range_color=[0, color_cap],
    color_continuous_scale='Viridis',
    projection="natural earth",
    hover_name='city_name' if 'city_name' in data.columns else None,
    hover_data=hover_cols,
    title=f"Geographic Price Distribution (Color capped at 95th percentile: €{color_cap:.0f})"
)

fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

### Spatial Interpretation of Price Distribution
- **Spatial Clustering of Luxury:** The map visually confirms our earlier statistical findings regarding right-skewness. The highest-priced listings (bright yellow points representing the top 5%) are not randomly scattered; they form tight geographic clusters strictly concentrated in the city center and prime tourist/coastal areas.
- **Suburban Affordability:** Standard and budget-friendly accommodations (darker points) dominate the peripheral neighborhoods, reflecting a clear distance-to-center price gradient.